# 🎬 Netflix Titles — Exploratory Data Analysis

An end-to-end EDA of the Netflix Movies & TV Shows dataset (8,800+ titles), exploring content trends, global production patterns, genres, ratings, and platform growth over time.

**Dataset:** [Netflix Movies and TV Shows (Kaggle)](https://www.kaggle.com/datasets/shivamb/netflix-shows)


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
%matplotlib inline


## 2. Load the Data

In [ ]:
df = pd.read_csv("../data/netflix_titles.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum().sort_values(ascending=False)


## 3. Data Cleaning

Steps taken:
- Fill missing `director`, `cast`, `country` with `"Not Specified"`
- Fill missing `rating` with the mode
- Drop rows with missing `date_added` (very small number, needed for time-based analysis)
- Parse `date_added` into a proper datetime and extract `year_added` / `month_added`
- Extract a `primary_country` (first country listed) for cleaner grouping
- Extract numeric `duration_int` from the `duration` column


In [ ]:
df["director"] = df["director"].fillna("Not Specified")
df["cast"] = df["cast"].fillna("Not Specified")
df["country"] = df["country"].fillna("Not Specified")
df["rating"] = df["rating"].fillna(df["rating"].mode()[0])
df = df.dropna(subset=["date_added"]).drop_duplicates()

df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()
df["primary_country"] = df["country"].apply(lambda x: x.split(",")[0].strip())
df["duration_int"] = df["duration"].str.extract(r"(\d+)").astype(float)

print("Cleaned shape:", df.shape)
df.head()


## 4. Movies vs TV Shows

In [ ]:
type_counts = df["type"].value_counts()
plt.figure(figsize=(6,6))
plt.pie(type_counts, labels=type_counts.index, autopct="%1.1f%%",
        colors=["#E50914", "#221f1f"], startangle=90,
        textprops={"color":"white","fontsize":12,"weight":"bold"},
        wedgeprops={"edgecolor":"white","linewidth":2})
plt.title("Movies vs TV Shows on Netflix", fontsize=14, weight="bold")
plt.show()


**Insight:** Movies make up roughly 70% of Netflix's catalog, with TV Shows at about 30%.

## 5. Content Growth Over Time

In [ ]:
yearly = df.groupby(["year_added","type"]).size().unstack(fill_value=0)
yearly = yearly[yearly.index <= 2021]
yearly.plot(kind="line", marker="o", figsize=(10,5), color=["#221f1f","#E50914"], linewidth=2.5)
plt.title("Content Added to Netflix by Year", fontsize=14, weight="bold")
plt.xlabel("Year Added"); plt.ylabel("Number of Titles")
plt.show()


**Insight:** Content additions grew rapidly between 2015–2019, peaking in 2019, then dipped slightly — likely tied to the pandemic slowing production/licensing in 2020-2021.

## 6. Top Producing Countries

In [ ]:
top_countries = df[df["primary_country"] != "Not Specified"]["primary_country"].value_counts().head(10)
plt.figure(figsize=(9,6))
sns.barplot(x=top_countries.values, y=top_countries.index, hue=top_countries.index, palette="Reds_r", legend=False)
plt.title("Top 10 Countries by Number of Titles Produced", fontsize=14, weight="bold")
plt.xlabel("Number of Titles"); plt.ylabel("Country")
plt.show()


**Insight:** The United States dominates the catalog by a wide margin, followed by India and the United Kingdom.

## 7. Top Genres

In [ ]:
all_genres = df["listed_in"].str.split(", ").explode()
top_genres = all_genres.value_counts().head(10)
plt.figure(figsize=(9,6))
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index, palette="rocket", legend=False)
plt.title("Top 10 Genres on Netflix", fontsize=14, weight="bold")
plt.xlabel("Number of Titles"); plt.ylabel("Genre")
plt.show()


## 8. Content Rating Distribution

In [ ]:
rating_counts = df["rating"].value_counts().head(10)
plt.figure(figsize=(10,5))
sns.barplot(x=rating_counts.index, y=rating_counts.values, hue=rating_counts.index, palette="mako", legend=False)
plt.title("Distribution of Content Ratings", fontsize=14, weight="bold")
plt.xticks(rotation=45)
plt.xlabel("Rating"); plt.ylabel("Number of Titles")
plt.show()


**Insight:** TV-MA (mature audiences) is the most common rating, indicating Netflix's catalog skews toward adult content.

## 9. Movie Duration Analysis

In [ ]:
movies = df[df["type"] == "Movie"]
plt.figure(figsize=(9,5))
sns.histplot(movies["duration_int"].dropna(), bins=30, color="#E50914", kde=True)
plt.axvline(movies["duration_int"].mean(), color="black", linestyle="--",
            label=f"Mean = {movies['duration_int'].mean():.0f} min")
plt.title("Distribution of Movie Durations", fontsize=14, weight="bold")
plt.xlabel("Duration (minutes)"); plt.legend()
plt.show()


**Insight:** Most movies run 80–120 minutes, with an average runtime of ~100 minutes.

## 10. TV Show Seasons

In [ ]:
tv = df[df["type"] == "TV Show"]
season_counts = tv["duration_int"].value_counts().sort_index().head(10)
plt.figure(figsize=(9,5))
sns.barplot(x=season_counts.index.astype(int), y=season_counts.values,
            hue=season_counts.index.astype(int), palette="Reds", legend=False)
plt.title("TV Shows: Number of Seasons", fontsize=14, weight="bold")
plt.xlabel("Number of Seasons"); plt.ylabel("Number of Shows")
plt.show()


**Insight:** The vast majority of TV shows on Netflix have just 1 season — long-running multi-season shows are rare.

## 11. Seasonality: When Does Netflix Add Content?

In [ ]:
month_order = ["January","February","March","April","May","June","July",
               "August","September","October","November","December"]
heat = df.groupby(["month_added","year_added"]).size().unstack(fill_value=0)
heat = heat.reindex(month_order)
heat = heat[[c for c in heat.columns if 2015 <= c <= 2021]]
plt.figure(figsize=(11,6))
sns.heatmap(heat, cmap="Reds", linewidths=0.5, cbar_kws={"label": "Titles Added"})
plt.title("Content Additions: Month vs Year", fontsize=14, weight="bold")
plt.show()


## 12. Top Directors

In [ ]:
directors = df[df["director"] != "Not Specified"]["director"].str.split(", ").explode()
top_directors = directors.value_counts().head(10)
plt.figure(figsize=(9,6))
sns.barplot(x=top_directors.values, y=top_directors.index, hue=top_directors.index, palette="flare", legend=False)
plt.title("Top 10 Directors by Number of Titles", fontsize=14, weight="bold")
plt.xlabel("Number of Titles"); plt.ylabel("Director")
plt.show()


## 13. Key Takeaways

1. **Movies dominate** the catalog (~70%) over TV Shows (~30%).
2. **Explosive growth 2015–2019**, peaking in 2019, then a slight pandemic-era slowdown.
3. **The US, India, and UK** are the top content-producing countries.
4. **International Movies, Dramas, and Comedies** are the most common genres.
5. **TV-MA is the most frequent rating** — the catalog leans toward mature audiences.
6. **Average movie length is ~100 minutes**; most TV shows are single-season.
7. Netflix tends to **add the most content in December and January** (holiday season releases).

---
*Analysis by [Your Name] — built with pandas, matplotlib & seaborn.*
